# 05 — Ensemble Evaluation

Evaluate the full IF + Prophet + RiskScorer pipeline.
Verify score distributions and risk threshold calibration.


In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings; warnings.filterwarnings('ignore')

model  = joblib.load('../artifacts/isolation_forest_v1.joblib')
scaler = joblib.load('../artifacts/scaler_v1.joblib')
print('Models loaded.')


In [ ]:
# Score distribution
df = pd.read_csv('../data/raw/hot_corridor.csv')
FEATURES = ['inlet_temp_c','outlet_temp_c','power_kw','airflow_cfm','humidity_pct','cpu_util_pct']
avail = [c for c in FEATURES if c in df.columns]
X = df[avail].fillna(df[avail].median())
X_scaled = scaler.transform(X)
scores = model.decision_function(X_scaled)

print(f'IF score  mean={scores.mean():.4f}  std={scores.std():.4f}')
print(f'           min={scores.min():.4f}  max={scores.max():.4f}')


In [ ]:
# Ensemble risk formula
import math
W_IF, W_FC, W_FREQ = 0.50, 0.35, 0.15

def norm_if(s): return 1/(1+math.exp(-s*10))

sample = float(scores[len(scores)//2])
risk = min(100, max(0, (W_IF*norm_if(sample) + W_FC*0.1 + W_FREQ*0.0)*100))
label = 'healthy' if risk<35 else 'at_risk' if risk<65 else 'critical'
print(f'Sample risk={risk:.1f}  label={label}')


## Risk Thresholds (calibrated)

| Score | Status | Color |
|-------|--------|-------|
| 0–35 | Healthy | Green |
| 35–65 | At Risk | Amber |
| 65–100 | Critical | Red |

**Platform ready.** Run: `docker-compose up`
